# 2. 단일 Lakehouse 비교 테스트: 온톨로지 생성

[첫 번째 노트북](01_copy_to_single_lakehouse.ipynb)의 복사 검증이 끝난 뒤 실행합니다. 해당 노트북의 `Files/test_setup/manifest.json`을 데이터 계약으로 읽습니다. 다른 노트북의 Python 변수나 실행 세션에 의존하지 않습니다.

- **첫 번째 노트북이 만든 테스트 Lakehouse를 기본으로 Attach**하고 아래 `TEST_LAKEHOUSE_ID`를 입력합니다.
- UI4의 엔터티 4개(Equipment, ProcessRun, Inspection, Nonconformance), 관계 5개, 키·속성 ID·이름·타입을 유지합니다. 정적 바인딩과 관계 매핑의 위치만 테스트 Lakehouse의 `dbo`로 바꿉니다. FDC는 포함하지 않습니다.
- Contributor 이상, Ontology Preview 설정 및 Fabric 용량이 필요합니다. 권한·테넌트 설정을 자동으로 완화하지 않습니다. NotebookUtils 실행자 인증과 Fabric 기본 `requests`만 사용합니다.
- 기본 `DRY_RUN = True`로 계획을 확인합니다. 생성하려면 False로 바꾸고 설정 셀부터 재실행합니다. 동명 온톨로지는 덮어쓰지 않습니다.
- 생성과 질의 성공은 다릅니다. 이 노트북은 생성 정의를 검증하며, **Graph 로드 완료 후 브라우저 새로 고침 및 새 Queryset**에서 마지막 셀의 관계 쿼리를 각각 실행해야 테스트가 끝납니다.
- 이 비교는 저장 위치뿐 아니라 Delta 파일 재작성도 포함합니다. 성공만으로 '다른 Lakehouse는 미지원'이라고 결론내리지 않습니다.

## 비용과 정리

온톨로지 및 Fabric 관리 Graph/보조 항목과 전체 적재 비용이 발생합니다. 기존 MES/QMS/FDC와 UI4는 변경하지 않습니다. 테스트 종료 후 출력된 ID를 대조해 **이번 테스트 온톨로지와 하위 항목, 첫 번째 노트북의 테스트 Lakehouse만** UI에서 정리합니다. 자동 삭제 셀은 없습니다. 원본 `mes_lakehouse`, `qms_lakehouse`와 기존 그래프는 삭제하지 않습니다.

In [ ]:
import base64
import copy
import json
import re
import time
import uuid
from urllib.parse import urlsplit

import notebookutils
import requests
from pyspark.sql.types import StructType

TEST_LAKEHOUSE_ID = '<TEST_LAKEHOUSE_ID_FROM_NOTEBOOK_01>'
ONTOLOGY_NAME = 'MES_QMS_SINGLE_TEST'
DRY_RUN = True
WORKSPACE_ID = str(uuid.UUID(notebookutils.runtime.context['currentWorkspaceId']))
API_ROOT = 'https://api.fabric.microsoft.com/v1'


def api_call(method, path, body=None):
    url = API_ROOT + path if path.startswith('/') else path
    parsed = urlsplit(url)
    if parsed.scheme != 'https' or parsed.netloc != 'api.fabric.microsoft.com' or not parsed.path.startswith('/v1/'):
        raise ValueError('Unexpected Fabric API URL')
    for attempt in range(5):
        try:
            response = requests.request(method, url, json=body, headers={
                'Authorization': 'Bearer ' + notebookutils.credentials.getToken('pbi'),
            }, timeout=90, allow_redirects=False)
        except requests.RequestException:
            raise RuntimeError('Fabric transport failure; inspect existing items before retrying a create.') from None
        with response:
            headers = dict(response.headers)
            payload = response.json() if response.content else None
            status_code = response.status_code
        if status_code == 429 and attempt < 4:
            time.sleep(min(float(headers.get('Retry-After', 10)), 60))
            continue
        if status_code not in (200, 201, 202, 204):
            raise RuntimeError(f'Fabric HTTP {status_code}: {json.dumps(payload, ensure_ascii=False)}')
        return status_code, payload, headers
    raise RuntimeError('Fabric throttling retry limit exceeded')


def operation(method, path, body=None):
    status_code, payload, headers = api_call(method, path, body)
    if status_code != 202:
        return payload
    operation_id = str(uuid.UUID(headers['x-ms-operation-id']))
    deadline = time.monotonic() + 1800
    while time.monotonic() < deadline:
        time.sleep(min(float(headers.get('Retry-After', 5)), 60))
        _, state, headers = api_call('GET', f'/operations/{operation_id}')
        if state['status'] == 'Succeeded':
            return api_call('GET', f'/operations/{operation_id}/result')[1]
        if state['status'] not in ('NotStarted', 'Running'):
            raise RuntimeError(f'Fabric operation failed: {state}')
    raise TimeoutError('Operation completion unknown; inspect the item before retrying.')


def list_items(path):
    rows, visited = [], set()
    while path:
        if path in visited:
            raise RuntimeError('Repeated pagination URL')
        visited.add(path)
        payload = api_call('GET', path)[1]
        rows.extend(payload['value'])
        path = payload.get('continuationUri')
        if payload.get('continuationToken') and not path:
            raise RuntimeError('Pagination URI missing; refusing an incomplete list')
    return rows


def decode_definition(definition):
    parts = {}
    for part in definition['parts']:
        if part['payloadType'] != 'InlineBase64' or part['path'] in parts:
            raise ValueError('Unsupported or duplicate definition part')
        parts[part['path']] = json.loads(base64.b64decode(part['payload'], validate=True))
    return parts


def remap_definition(baseline, workspace_id, lakehouse_id, ontology_name, allowed_tables):
    parts = copy.deepcopy(baseline)
    for path, body in parts.items():
        source = None
        if path.startswith('EntityTypes/') and '/DataBindings/' in path:
            config = body['dataBindingConfiguration']
            if config['dataBindingType'] != 'NonTimeSeries':
                raise ValueError('Only UI4 static bindings are allowed')
            source = config['sourceTableProperties']
        elif path.startswith('RelationshipTypes/') and '/Contextualizations/' in path:
            source = body['dataBindingTable']
        if source is not None:
            if source['sourceType'] != 'LakehouseTable' or source['sourceTableName'] not in allowed_tables:
                raise ValueError('Unexpected source in baseline')
            source.update(workspaceId=workspace_id, itemId=lakehouse_id, sourceSchema='dbo')
    parts['.platform'] = {'metadata': {'type': 'Ontology', 'displayName': ontology_name}}
    return parts


def normalized(value):
    if isinstance(value, dict):
        return {key: normalized(item) for key, item in sorted(value.items())
                if key != '$schema' and item is not None and item != [] and item != {}}
    if isinstance(value, list):
        return sorted((normalized(item) for item in value), key=lambda item: json.dumps(item, sort_keys=True))
    return value


In [ ]:
from pyspark.sql import functions as F

TEST_LAKEHOUSE_ID = str(uuid.UUID(TEST_LAKEHOUSE_ID))
context = notebookutils.runtime.context
if context.get('defaultLakehouseId') != TEST_LAKEHOUSE_ID or context.get('defaultLakehouseWorkspaceId') != WORKSPACE_ID:
    raise ValueError('Attach the verified test Lakehouse as the default')
if not re.fullmatch(r'[A-Za-z][A-Za-z0-9_]{0,79}', ONTOLOGY_NAME) or type(DRY_RUN) is not bool:
    raise ValueError('Invalid ontology name or DRY_RUN setting')
SETUP_ROOT = f'abfss://{WORKSPACE_ID}@onelake.dfs.fabric.microsoft.com/{TEST_LAKEHOUSE_ID}/Files/test_setup'
MANIFEST = json.loads(notebookutils.fs.head(SETUP_ROOT + '/manifest.json', 1048576))
if MANIFEST.get('format_version') != 1 or (MANIFEST['workspace_id'], MANIFEST['test_lakehouse_id']) != (WORKSPACE_ID, TEST_LAKEHOUSE_ID):
    raise ValueError('Manifest workspace or test Lakehouse mismatch')
if TEST_LAKEHOUSE_ID in (MANIFEST['mes_id'], MANIFEST['qms_id']):
    raise ValueError('Refusing to use an original Lakehouse')
TABLE_KEYS = {'mes_equipment': 'eqp_id', 'mes_process_result': 'id', 'qms_inspection': 'inspection_id', 'qms_nonconformance': 'ncr_id'}
if set(MANIFEST['tables']) != set(TABLE_KEYS):
    raise ValueError('Expected exactly four copied tables')
metadata = api_call('GET', f'/workspaces/{WORKSPACE_ID}/lakehouses/{TEST_LAKEHOUSE_ID}')[1]
if metadata['properties'].get('defaultSchema') != 'dbo':
    raise ValueError('Test Lakehouse must use dbo')

frames = {}
for table, key in TABLE_KEYS.items():
    path = f'abfss://{WORKSPACE_ID}@onelake.dfs.fabric.microsoft.com/{TEST_LAKEHOUSE_ID}/Tables/dbo/{table}'
    frame = spark.read.format('delta').load(path)
    report = MANIFEST['tables'][table]
    if frame.schema != StructType.fromJson(report['schema']) or frame.count() != report['rows']:
        raise ValueError(f'{table}: schema or row count changed since copy')
    if frame.filter(F.col(key).isNull()).limit(1).count() or frame.select(key).distinct().count() != report['rows']:
        raise ValueError(f'{table}: invalid primary key')
    detail = spark.sql(f'DESCRIBE DETAIL delta.`{path}`').first().asDict()
    if detail['properties'].get('delta.columnMapping.mode', 'none') != 'none':
        raise ValueError(f'{table}: column mapping enabled')
    frames[table] = frame

BASELINE = MANIFEST['definition']
entity_bodies = {body['id']: body for path, body in BASELINE.items()
                 if path.startswith('EntityTypes/') and path.endswith('/definition.json')}
relation_bodies = {body['id']: body for path, body in BASELINE.items()
                   if path.startswith('RelationshipTypes/') and path.endswith('/definition.json')}
if len(entity_bodies) != 4 or len(relation_bodies) != 5:
    raise ValueError('Invalid UI4 baseline')
entity_tables = {}
for path, body in BASELINE.items():
    if path.startswith('EntityTypes/') and '/DataBindings/' in path:
        entity = entity_bodies[path.split('/')[1]]
        config = body['dataBindingConfiguration']
        table = config['sourceTableProperties']['sourceTableName']
        if table not in TABLE_KEYS or any(mapping['sourceColumnName'] not in frames[table].columns for mapping in config['propertyBindings']):
            raise ValueError('Missing bound source column')
        entity_tables[entity['id']] = table

for path, binding in BASELINE.items():
    if not path.startswith('RelationshipTypes/') or '/Contextualizations/' not in path:
        continue
    relation = relation_bodies[path.split('/')[1]]
    mapping = frames[binding['dataBindingTable']['sourceTableName']]
    valid = mapping
    for side in ('source', 'target'):
        entity_id = relation[side]['entityTypeId']
        node_table = entity_tables[entity_id]
        node = frames[node_table]
        references = binding[side + 'KeyRefBindings']
        if len(references) != 1 or references[0]['targetPropertyId'] != entity_bodies[entity_id]['entityIdParts'][0]:
            raise ValueError('Unexpected entity key binding')
        column = references[0]['sourceColumnName']
        key = TABLE_KEYS[node_table]
        if mapping.schema[column].dataType != node.schema[key].dataType:
            raise ValueError(f"{relation['name']}: key types differ")
        if mapping.filter(F.col(column).isNotNull()).join(node, mapping[column] == node[key], 'left_anti').limit(1).count():
            raise ValueError(f"{relation['name']}: missing referenced node")
        valid = valid.filter(F.col(column).isNotNull())
    if valid.count() != MANIFEST['expected_counts'][relation['name']]:
        raise ValueError(f"{relation['name']}: expected relationship count changed")

DESIRED = remap_definition(BASELINE, WORKSPACE_ID, TEST_LAKEHOUSE_ID, ONTOLOGY_NAME, TABLE_KEYS)
print('DATA VERIFIED:', {table: report['rows'] for table, report in MANIFEST['tables'].items()})
print('PLAN:', ONTOLOGY_NAME, '4 entities / 5 relationships / one Lakehouse:', TEST_LAKEHOUSE_ID)
print('Expected relationship counts:', MANIFEST['expected_counts'])
print('DRY_RUN:', DRY_RUN)


In [ ]:
ONTOLOGY_ID = None
if DRY_RUN:
    print('DRY RUN: no ontology was created. Set DRY_RUN=False and rerun cells 2 onward.')
else:
    existing = list_items(f'/workspaces/{WORKSPACE_ID}/ontologies')
    if any(item['displayName'].casefold() == ONTOLOGY_NAME.casefold() for item in existing):
        raise RuntimeError('Ontology name already exists; inspect it or choose a new name. No update/delete is performed.')
    definition = {'parts': [
        {'path': path, 'payload': base64.b64encode(json.dumps(body, ensure_ascii=False).encode()).decode(), 'payloadType': 'InlineBase64'}
        for path, body in DESIRED.items()
    ]}
    created = operation('POST', f'/workspaces/{WORKSPACE_ID}/ontologies', {
        'displayName': ONTOLOGY_NAME,
        'description': 'UI4 comparison using copied tables in one Lakehouse.',
        'definition': definition,
    })
    ONTOLOGY_ID = str(uuid.UUID(created['id']))
    print('Created ontology:', ONTOLOGY_ID)
    saved = decode_definition(operation('POST', f'/workspaces/{WORKSPACE_ID}/ontologies/{ONTOLOGY_ID}/getDefinition')['definition'])
    expected = {path: body for path, body in DESIRED.items() if path != '.platform'}
    actual = {path: body for path, body in saved.items() if path != '.platform'}
    if normalized(actual) != normalized(expected):
        raise RuntimeError('Saved definition differs from the requested definition; inspect before retrying')
    notebookutils.fs.put(SETUP_ROOT + f'/ontology_{ONTOLOGY_ID}.json', json.dumps({
        'ontology_id': ONTOLOGY_ID, 'name': ONTOLOGY_NAME, 'workspace_id': WORKSPACE_ID,
        'test_lakehouse_id': TEST_LAKEHOUSE_ID, 'definition_verified': True,
        'query_verified': False, 'expected_counts': MANIFEST['expected_counts'],
    }, ensure_ascii=False), False)
    print('DEFINITION VERIFIED. This does not establish relationship query success.')
    print(f'https://app.fabric.microsoft.com/groups/{WORKSPACE_ID}/ontologies/{ONTOLOGY_ID}')


In [ ]:
def wait_job(path):
    deadline = time.monotonic() + 1800
    previous = None
    while time.monotonic() < deadline:
        _, job, headers = api_call('GET', path)
        if job.get('status') != previous:
            print('Graph job:', job.get('id'), job.get('status'))
            previous = job.get('status')
        if job['status'] in ('Completed', 'Failed', 'Cancelled', 'Deduped'):
            return job
        if job['status'] not in ('NotStarted', 'InProgress'):
            raise RuntimeError(f'Unexpected Graph job state: {job}')
        time.sleep(min(float(headers.get('Retry-After', 15)), 60))
    raise TimeoutError('Graph job is still pending. Inspect its status; do not submit repeated refreshes.')


GRAPH_ID = None
if DRY_RUN:
    print('DRY RUN: Graph refresh skipped')
else:
    if not ONTOLOGY_ID:
        raise RuntimeError('Run the ontology creation/verification cell first')
    graph_name = f"{ONTOLOGY_NAME}_graph_{ONTOLOGY_ID.replace('-', '')}"
    deadline = time.monotonic() + 600
    while time.monotonic() < deadline:
        matches = [item for item in list_items(f'/workspaces/{WORKSPACE_ID}/graphModels') if item['displayName'] == graph_name]
        if len(matches) == 1:
            GRAPH_ID = str(uuid.UUID(matches[0]['id']))
            break
        if len(matches) > 1:
            raise RuntimeError('Multiple matching Graph items; inspect the ontology in Fabric')
        time.sleep(10)
    if GRAPH_ID is None:
        raise TimeoutError(f'Ontology {ONTOLOGY_ID} exists, but its Graph was not found by the expected name. Inspect it in Fabric.')
    print('Graph selected by exact generated name; this is not a parent-link verification:', GRAPH_ID)
    jobs_path = f'/workspaces/{WORKSPACE_ID}/items/{GRAPH_ID}/jobs/instances'
    completed = False
    for attempt in range(2):
        active = [job for job in list_items(jobs_path) if job['status'] in ('NotStarted', 'InProgress')]
        for job in active:
            result = wait_job(jobs_path + '/' + str(uuid.UUID(job['id'])))
            if result['status'] not in ('Completed', 'Deduped'):
                raise RuntimeError(f'Existing Graph load failed: {result}')
        status_code, payload, headers = api_call('POST', f'/workspaces/{WORKSPACE_ID}/graphModels/{GRAPH_ID}/jobs/refreshGraph/instances')
        if status_code == 200:
            result = {'status': 'Completed', 'itemId': GRAPH_ID}
        elif status_code == 202:
            location = headers.get('Location', '')
            if not location.startswith(API_ROOT + jobs_path + '/'):
                raise RuntimeError('Unexpected Graph job Location; inspect the submitted job')
            result = wait_job(location)
            if result.get('itemId') != GRAPH_ID:
                raise RuntimeError('Graph job belongs to a different item')
        else:
            raise RuntimeError(f'Unexpected refresh response: {status_code}, {payload}')
        if result['status'] == 'Completed':
            completed = True
            break
        if result['status'] != 'Deduped':
            raise RuntimeError(f'Graph refresh failed: {result}')
    if not completed:
        raise RuntimeError('Refresh was skipped due to a concurrent job. Wait and rerun only this cell.')
    notebookutils.fs.put(SETUP_ROOT + f'/refresh_{ONTOLOGY_ID}_{uuid.uuid4().hex}.json', json.dumps(result), False)
    print('GRAPH JOB COMPLETED; relation queries are still unverified.')
    print(f'https://app.fabric.microsoft.com/groups/{WORKSPACE_ID}/graph/{GRAPH_ID}/modeling')
    print('Refresh the browser and create a NEW Queryset after this completion. Do not reuse an earlier Queryset.')


## 최종 검증: 로드 완료 후 새 Queryset에서 한 블록씩 실행

Graph 모델 화면의 최신 저장 내용이 로드됐는지 확인하고 브라우저를 새로 고친 뒤 **새 Queryset**을 생성합니다. 기존 UI4 Graph가 아니라 바로 위 셀에서 출력한 새 Graph ID를 사용합니다. 조회 오류와 0건을 구분하고 오류 원문·Graph ID·시각을 기록합니다.

```text
MATCH (run:ProcessRun)-[:RunHasInspection]->(inspection:Inspection)
RETURN count(*) AS edge_count
```

```text
MATCH (equipment:Equipment)-[:EquipmentHasInspection]->(inspection:Inspection)
RETURN count(*) AS edge_count
```

```text
MATCH (equipment:Equipment)-[:EquipmentHasRuns]->(run:ProcessRun)
RETURN count(*) AS edge_count
```

```text
MATCH (equipment:Equipment)-[:EquipmentHasNcr]->(ncr:Nonconformance)
RETURN count(*) AS edge_count
```

```text
MATCH (inspection:Inspection)-[:InspectionHasNcr]->(ncr:Nonconformance)
RETURN count(*) AS edge_count
```

위 결과를 계획 셀의 `expected_counts`와 비교합니다. 앞서 확인한 원본 기준 참고값은 순서대로 **131, 136, 84, 34, 49**이며, 새로 복사할 때 계산한 값이 실제 판정 기준입니다. 키 직접 비교 결과로 관계 조회 성공을 대신하지 않습니다.

- 모두 일치: 이번 단일 Lakehouse 구성에서는 관계 조회가 동작함. 복사에 따른 Delta 파일 재작성 등도 함께 달라졌으므로 Lakehouse 경계가 제품 제약이라고 단정하지 않음.
- 일부 실패: 해당 관계·오류·기대 건수를 보존하고 원본 교차 구성과 비교. 원본 테이블을 지우거나 권한을 임의로 넓히지 않음.
- 정리: 이번에 생성한 온톨로지 및 하위 Graph/보조 항목을 확인 후 정리하고, 마지막에 테스트 전용 Lakehouse를 삭제. 기본 Lakehouse가 원본으로 바뀔 수 있으므로 일반 `DROP TABLE` 명령은 제공하지 않음.

## 공식 자료

- [Lakehouse 생성](https://learn.microsoft.com/en-us/rest/api/fabric/lakehouse/items/create-lakehouse)
- [Ontology 생성](https://learn.microsoft.com/en-us/rest/api/fabric/ontology/items/create-ontology)
- [Ontology 정의 조회](https://learn.microsoft.com/en-us/rest/api/fabric/ontology/items/get-ontology-definition)
- [Graph 새로 고침](https://learn.microsoft.com/en-us/fabric/iq/ontology/how-to-view-entity-type-details#refresh-the-graph-model)
- [Queryset 갱신 문제](https://learn.microsoft.com/en-us/fabric/graph/troubleshooting-and-faq#queryset-doesnt-show-recent-model-changes)

로컬 JSON·문법·정의 변환 검증과 Fabric 실행 검증은 별개입니다. 이 노트북 자체의 원격 실행 및 실제 관계 쿼리 결과는 실행 후 확인해야 합니다.